# CVFS 3D Reconstruction Demo

This notebook demonstrates the current approximate 3D reconstruction workflow for the Computer Vision Freekick System.

The demo uses a lightweight sample CSV containing synchronized event points from the left camera and behind-goal camera views. Raw videos and YOLO model weights are excluded from the repository because of file size.

The reconstruction uses measured goal geometry, image-space reference points, and approximate perspective mapping to estimate the soccer ball trajectory in 3D space.

In [30]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

pio.renderers.default = "notebook_connected"

csv_path = Path("../data/sample/approx_cvfs_3d_points.csv")
df = pd.read_csv(csv_path)

df["_original_order"] = np.arange(len(df))

df.head()

,event,behind_frame,behind_x_px,behind_y_px,left_frame,left_x_px,left_y_px,x_m,y_m,z_m,_original_order
0,Precontact,17,1069.34,908.59,15,1261.50,354.78,0.108,-0.060,0.060,0
1,Contact,24,1071.10,906.77,285,1261.50,354.78,0.134,-0.060,0.060,1
2,First motion,25,1074.99,900.66,286,1273.46,354.66,0.191,-0.253,0.062,2
3,Separation,26,1081.24,888.14,287,1257.80,352.54,0.283,0.000,0.102,3
4,Early rise,58,1335.77,236.82,321,839.56,252.15,4.024,6.767,1.972,4


In [31]:
# Real measured field / goal geometry
goal_width = 7.265
goal_height = 2.52
goal_half_width = goal_width / 2

goal_y = 19.945
net_depth = 2.0
net_y = goal_y + net_depth

right_width = 3.565
left_width = goal_width - right_width

six_yard_depth = 5.4864
eighteen_yard_depth = 16.4592

y_goal_line = goal_y
y_six_line = goal_y - six_yard_depth
y_eighteen_line = goal_y - eighteen_yard_depth

print(f"Goal width: {goal_width} m")
print(f"Goal height: {goal_height} m")
print(f"Ball-to-goal distance: {goal_y} m")
print(f"Net depth: {net_depth} m")

Goal width: 7.265 m
Goal height: 2.52 m
Ball-to-goal distance: 19.945 m
Net depth: 2.0 m


In [32]:
# Left-view height references
left_post_bottom_y = 358.0
left_post_top_y = 227.0

# Behind-view reference points
R_goal = (1311.0, 236.0)
R_near = (1727.0, 397.0)

L_goal = (822.0, 230.0)
L_near = (4.0, 545.0)

C_goal = (1062.0, 237.0)
C_near = (1066.0, 1074.0)

goal_line_p1 = (828.0, 230.0)
goal_line_p2 = (1302.0, 233.0)

six_line_p1 = (714.0, 257.0)
six_line_p2 = (1865.0, 265.0)

eighteen_line_p1 = (0.0, 447.0)
eighteen_line_p2 = (1919.0, 486.0)

In [33]:
def line_from_points(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    
    A = y1 - y2
    B = x2 - x1
    C = x1 * y2 - x2 * y1
    
    return np.array([A, B, C], dtype=float)


def intersect_lines(L1, L2):
    A1, B1, C1 = L1
    A2, B2, C2 = L2
    
    D = A1 * B2 - A2 * B1
    
    if abs(D) < 1e-9:
        return None
    
    x = (B1 * C2 - B2 * C1) / D
    y = (C1 * A2 - C2 * A1) / D
    
    return np.array([x, y], dtype=float)

In [34]:
line_left = line_from_points(L_near, L_goal)
line_blue = line_from_points(C_goal, C_near)
line_right = line_from_points(R_goal, R_near)

line_goal = line_from_points(goal_line_p1, goal_line_p2)
line_six = line_from_points(six_line_p1, six_line_p2)
line_eighteen = line_from_points(eighteen_line_p1, eighteen_line_p2)

L_goal_int = intersect_lines(line_left, line_goal)
C_goal_int = intersect_lines(line_blue, line_goal)
R_goal_int = intersect_lines(line_right, line_goal)

L_six_int = intersect_lines(line_left, line_six)
C_six_int = intersect_lines(line_blue, line_six)
R_six_int = intersect_lines(line_right, line_six)

L_eighteen_int = intersect_lines(line_left, line_eighteen)
C_eighteen_int = intersect_lines(line_blue, line_eighteen)
R_eighteen_int = intersect_lines(line_right, line_eighteen)

required_intersections = [
    L_goal_int, C_goal_int, R_goal_int,
    L_six_int, C_six_int, R_six_int,
    L_eighteen_int, C_eighteen_int, R_eighteen_int,
]

if any(p is None for p in required_intersections):
    raise ValueError("One or more reference-line intersections failed. Check reference points.")

depth_world = np.array([y_eighteen_line, y_six_line, y_goal_line], dtype=float)

left_x_img = np.array([L_eighteen_int[0], L_six_int[0], L_goal_int[0]], dtype=float)
blue_x_img = np.array([C_eighteen_int[0], C_six_int[0], C_goal_int[0]], dtype=float)
right_x_img = np.array([R_eighteen_int[0], R_six_int[0], R_goal_int[0]], dtype=float)

left_x_poly = np.poly1d(np.polyfit(depth_world, left_x_img, 2))
blue_x_poly = np.poly1d(np.polyfit(depth_world, blue_x_img, 2))
right_x_poly = np.poly1d(np.polyfit(depth_world, right_x_img, 2))

print("Depth anchors in meters:", depth_world)
print("Left image x anchors:", left_x_img)
print("Center image x anchors:", blue_x_img)
print("Right image x anchors:", right_x_img)

Depth anchors in meters: [ 3.4858 14.4586 19.945 ]
Left image x anchors: [245.53087351 751.21403163 822.09701906]
Center image x anchors: [1063.10683685 1062.10714223 1061.97362411]
Right image x anchors: [1959.0667088  1377.17075221 1303.2692031 ]


In [35]:
origin = df[df["event"].str.lower() == "precontact"].iloc[0]
hits_net = df[df["event"].str.lower() == "hits net"].iloc[0]

# Recompute forward distance y from the left view
left_start_x = origin["left_x_px"]
left_goal_x = hits_net["left_x_px"]

df["y_plot"] = net_y * (left_start_x - df["left_x_px"]) / (left_start_x - left_goal_x)

precontact_y = df.loc[df["event"].str.lower() == "precontact", "y_plot"].iloc[0]
df["y_plot"] = df["y_plot"] - precontact_y

# Recompute vertical height z from the left view
df["z_plot"] = goal_height * (left_post_bottom_y - df["left_y_px"]) / (
    left_post_bottom_y - left_post_top_y
)

precontact_z = df.loc[df["event"].str.lower() == "precontact", "z_plot"].iloc[0]
df["z_plot"] = df["z_plot"] - precontact_z
df["z_plot"] = df["z_plot"].clip(lower=0)

# Recompute lateral x from the behind view
depth_for_x = df["y_plot"].clip(lower=0, upper=goal_y)

u_left = left_x_poly(depth_for_x)
u_blue = blue_x_poly(depth_for_x)
u_right = right_x_poly(depth_for_x)

u_ball = df["behind_x_px"].to_numpy()

x_vals = []

for ub, ul, uc, ur in zip(u_ball, u_left, u_blue, u_right):
    if ub >= uc:
        frac_right = (ub - uc) / (ur - uc)
        x_val = frac_right * right_width
    else:
        frac_left = (uc - ub) / (uc - ul)
        x_val = -frac_left * left_width
    
    x_vals.append(x_val)

df["x_plot"] = np.array(x_vals)

precontact_x = df.loc[df["event"].str.lower() == "precontact", "x_plot"].iloc[0]
df["x_plot"] = df["x_plot"] - precontact_x

df[["event", "x_plot", "y_plot", "z_plot"]]

,event,x_plot,y_plot,z_plot
0,Precontact,0.000000,0.000000,0.000000
1,Contact,0.005222,0.000000,0.000000
2,First motion,0.016765,-0.210647,0.002308
3,Separation,0.035616,0.065167,0.043090
4,Early rise,1.552198,7.431478,1.974256
5,Mid rise,2.665940,11.473756,2.799316
6,Late rise,3.422059,14.122169,3.104986
7,Apex,3.844490,16.088973,3.142690
8,Early fall,4.013440,18.034113,2.938974
9,Mid fall,3.937740,19.600229,2.493069


In [36]:
hits_net_row = df[df["event"].str.lower() == "hits net"].iloc[0]

print("=== PRECONTACT USED AS ORIGIN ===")
print("x = 0.000, y = 0.000, z = 0.000")
print()

print("=== HITS NET RELATIVE TO PRECONTACT ===")
print(f"x = {hits_net_row['x_plot']:.3f} m")
print(f"y = {hits_net_row['y_plot']:.3f} m")
print(f"z = {hits_net_row['z_plot']:.3f} m")
print()

print("=== CLEARANCE RELATIVE TO RIGHT POST AND CROSSBAR ===")
print(f"Inside right post by: {goal_half_width - hits_net_row['x_plot']:.3f} m")
print(f"Below crossbar by:    {goal_height - hits_net_row['z_plot']:.3f} m")
print()

print("=== RECONSTRUCTED EVENT POINTS ===")
print(df[["event", "x_plot", "y_plot", "z_plot"]].to_string(index=False))

=== PRECONTACT USED AS ORIGIN ===
x = 0.000, y = 0.000, z = 0.000

=== HITS NET RELATIVE TO PRECONTACT ===
x = 3.308 m
y = 21.945 m
z = 1.170 m

=== CLEARANCE RELATIVE TO RIGHT POST AND CROSSBAR ===
Inside right post by: 0.325 m
Below crossbar by:    1.350 m

=== RECONSTRUCTED EVENT POINTS ===
       event   x_plot    y_plot   z_plot
  Precontact 0.000000  0.000000 0.000000
     Contact 0.005222  0.000000 0.000000
First motion 0.016765 -0.210647 0.002308
  Separation 0.035616  0.065167 0.043090
  Early rise 1.552198  7.431478 1.974256
    Mid rise 2.665940 11.473756 2.799316
   Late rise 3.422059 14.122169 3.104986
        Apex 3.844490 16.088973 3.142690
  Early fall 4.013440 18.034113 2.938974
    Mid fall 3.937740 19.600229 2.493069
   Late fall 3.607902 20.876617 1.886922
    Hits net 3.307833 21.945000 1.169973


In [37]:
if "frame_idx" in df.columns:
    df_plot = df.sort_values("frame_idx").reset_index(drop=True)
elif "frame" in df.columns:
    df_plot = df.sort_values("frame").reset_index(drop=True)
elif "time" in df.columns:
    df_plot = df.sort_values("time").reset_index(drop=True)
else:
    df_plot = df.sort_values("_original_order").reset_index(drop=True)

fit_df = df.copy().sort_values("y_plot").reset_index(drop=True)

apex_rows = fit_df[fit_df["event"].str.lower() == "apex"]

if len(apex_rows) == 0:
    raise ValueError("Could not find event named 'apex' in CSV.")

apex_idx = apex_rows.index[0]

pre_df = fit_df.loc[:apex_idx].copy()
post_df = fit_df.loc[apex_idx:].copy()

if len(pre_df) < 3:
    raise ValueError("Need at least 3 start-to-apex points for quadratic fit.")

if len(post_df) < 3:
    raise ValueError("Need at least 3 apex-to-net points for quadratic fit.")

y_pre = pre_df["y_plot"].to_numpy(dtype=float)
x_pre = pre_df["x_plot"].to_numpy(dtype=float)
z_pre = pre_df["z_plot"].to_numpy(dtype=float)

x_pre_poly = np.poly1d(np.polyfit(y_pre, x_pre, 2))
z_pre_poly = np.poly1d(np.polyfit(y_pre, z_pre, 2))

y_pre_guide = np.linspace(y_pre.min(), y_pre.max(), 150)
x_pre_guide = x_pre_poly(y_pre_guide)
z_pre_guide = np.clip(z_pre_poly(y_pre_guide), 0, None)

y_post = post_df["y_plot"].to_numpy(dtype=float)
x_post = post_df["x_plot"].to_numpy(dtype=float)
z_post = post_df["z_plot"].to_numpy(dtype=float)

x_post_poly = np.poly1d(np.polyfit(y_post, x_post, 2))
z_post_poly = np.poly1d(np.polyfit(y_post, z_post, 2))

y_post_guide = np.linspace(y_post.min(), y_post.max(), 150)
x_post_guide = x_post_poly(y_post_guide)
z_post_guide = np.clip(z_post_poly(y_post_guide), 0, None)

In [39]:
fig = go.Figure()

# Goal plane
fig.add_trace(go.Mesh3d(
    x=[-left_width, right_width, right_width, -left_width],
    y=[goal_y, goal_y, goal_y, goal_y],
    z=[0, 0, goal_height, goal_height],
    i=[0, 0],
    j=[1, 2],
    k=[2, 3],
    color="lightblue",
    opacity=0.25,
    showlegend=False,
    hoverinfo="skip",
))

# Back net plane
fig.add_trace(go.Mesh3d(
    x=[-left_width, right_width, right_width, -left_width],
    y=[net_y, net_y, net_y, net_y],
    z=[0, 0, goal_height, goal_height],
    i=[0, 0],
    j=[1, 2],
    k=[2, 3],
    color="lightblue",
    opacity=0.15,
    showlegend=False,
    hoverinfo="skip",
))

# Quadratic visual fits
fig.add_trace(go.Scatter3d(
    x=x_pre_guide,
    y=y_pre_guide,
    z=z_pre_guide,
    mode="lines",
    showlegend=False,
    line=dict(color="red", width=5, dash="dash"),
))

fig.add_trace(go.Scatter3d(
    x=x_post_guide,
    y=y_post_guide,
    z=z_post_guide,
    mode="lines",
    showlegend=False,
    line=dict(color="red", width=5, dash="dash"),
))

# Reconstructed event points
fig.add_trace(go.Scatter3d(
    x=df_plot["x_plot"],
    y=df_plot["y_plot"],
    z=df_plot["z_plot"],
    mode="markers",
    showlegend=False,
    marker=dict(color="red", size=6),
    text=df_plot["event"],
    hovertemplate="%{text}<br>x=%{x:.2f} m<br>y=%{y:.2f} m<br>z=%{z:.2f} m<extra></extra>",
))

# Goal frame
fig.add_trace(go.Scatter3d(
    x=[-left_width, -left_width, right_width, right_width, -left_width],
    y=[goal_y, goal_y, goal_y, goal_y, goal_y],
    z=[0, goal_height, goal_height, 0, 0],
    mode="lines",
    showlegend=False,
    line=dict(color="black", width=6),
))

# Center line
fig.add_trace(go.Scatter3d(
    x=[0, 0],
    y=[0, goal_y],
    z=[0, 0],
    mode="lines",
    showlegend=False,
    line=dict(color="blue", width=5),
))

# Side reference lines
fig.add_trace(go.Scatter3d(
    x=[-left_width, -left_width],
    y=[0, goal_y],
    z=[0, 0],
    mode="lines",
    showlegend=False,
    line=dict(color="goldenrod", width=4),
))

fig.add_trace(go.Scatter3d(
    x=[right_width, right_width],
    y=[0, goal_y],
    z=[0, 0],
    mode="lines",
    showlegend=False,
    line=dict(color="goldenrod", width=4),
))

# Back net frame
fig.add_trace(go.Scatter3d(
    x=[-left_width, -left_width, right_width, right_width, -left_width],
    y=[net_y, net_y, net_y, net_y, net_y],
    z=[0, goal_height, goal_height, 0, 0],
    mode="lines",
    showlegend=False,
    line=dict(color="lightblue", width=4),
))

for x_side in [-left_width, right_width]:
    fig.add_trace(go.Scatter3d(
        x=[x_side, x_side],
        y=[goal_y, net_y],
        z=[0, 0],
        mode="lines",
        showlegend=False,
        line=dict(color="lightblue", width=4),
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[x_side, x_side],
        y=[goal_y, net_y],
        z=[goal_height, goal_height],
        mode="lines",
        showlegend=False,
        line=dict(color="lightblue", width=4),
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[x_side, x_side],
        y=[net_y, net_y],
        z=[0, goal_height],
        mode="lines",
        showlegend=False,
        line=dict(color="lightblue", width=4),
    ))

fig.update_layout(
    title="Approximate 3D Reconstruction with Quadratic Visual Fit",
    showlegend=False,
    scene=dict(
        xaxis=dict(title="Lateral position x", range=[-5, 5], showticklabels=False),
        yaxis=dict(title="Forward position y", range=[0, net_y + 1], showticklabels=False),
        zaxis=dict(title="Height z", range=[0, 6], showticklabels=False),
        aspectratio=dict(x=1, y=2.5, z=0.7),
        camera=dict(eye=dict(x=1.2, y=-1.5, z=0.8)),
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

## Interpretation

This plot shows an approximate 3D reconstruction of the ball trajectory using synchronized event points from the left and behind camera views.

The red points are reconstructed event points. The dashed red curves are visual quadratic fits split around the apex. The goal and net geometry are drawn using measured field dimensions.

This is not full stereo triangulation. It is an approximate reconstruction based on measured goal geometry, camera-view reference lines, and selected synchronized event points.